# LLM serving: анализ реальных benchmark-запусков

ML Mentor · starter notebook · решения в репозитории отсутствуют.


## Статус: draft

Эта лабораторная анализирует только сохранённые результаты реальных запусков vLLM и SGLang на одинаковом оборудовании и workload. В репозитории пока нет подтверждённых fixtures, поэтому публиковать сравнительные числа или делать вывод о победителе нельзя.

Синтетические метрики запрещены. Черновик станет доступным учащимся только после записи настоящих запусков, проверки provenance и отдельного review.

**Будущий deliverable:** проверка сопоставимости запусков, throughput, TTFT и TPOT с p50/p95, график latency-throughput и инженерный вывод с перечислением ограничений.

In [ ]:
import hashlib
import json
from pathlib import Path

import pandas as pd


def find_repository_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "data" / "llm-serving-benchmarks" / "manifest.json").exists():
            return candidate
    raise FileNotFoundError("Запустите notebook из checkout репозитория ml-mentor-labs")


ROOT = find_repository_root()
BENCHMARK_DIR = ROOT / "data" / "llm-serving-benchmarks"
benchmark_manifest = json.loads((BENCHMARK_DIR / "manifest.json").read_text(encoding="utf-8"))
recorded_runs = benchmark_manifest["runs"]
BENCHMARK_READY = bool(recorded_runs)
assert all(run.get("provenance") == "recorded" for run in recorded_runs)


## 1. Проверьте provenance до анализа

Каждый run обязан указывать engine/version/commit, модель и revision, GPU, driver/CUDA, контейнер или lockfile, параметры workload, временную метку, сырой fixture и SHA-256. Запуски можно сравнивать только при совпадении полей из `comparabilityRequirements` протокола.

In [ ]:
def load_verified_runs(manifest: dict) -> pd.DataFrame:
    # TODO: если runs пуст, завершите работу без чисел; иначе проверьте SHA-256
    # каждого raw fixture и верните нормализованную таблицу реальных измерений.
    raise NotImplementedError


if not BENCHMARK_READY:
    raise RuntimeError("Нет проверенных реальных fixtures: сравнительные числа публиковать нельзя")

runs = load_verified_runs(benchmark_manifest)


## 2. Сравните только сопоставимые запуски

Покажите количество успешных запросов, input/output tokens, request throughput, output-token throughput, TTFT p50/p95 и TPOT p50/p95. Не объединяйте client-observed latency и внутренние engine-метрики под одним названием.

In [ ]:
# TODO: создайте comparison только из сопоставимых recorded runs.
comparison = None
LIMITATIONS = None


In [ ]:
# Self-check. В текущем draft он намеренно не проходит без реальных fixtures.
assert BENCHMARK_READY, "release gate: нужны реальные записанные запуски"
assert comparison is not None and len(comparison) >= 2
required = {"engine", "request_throughput", "output_token_throughput", "ttft_p50_ms", "ttft_p95_ms", "tpot_p50_ms", "tpot_p95_ms"}
assert required <= set(comparison.columns)
assert (comparison[list(required - {"engine"})] >= 0).all().all()
assert isinstance(LIMITATIONS, str) and LIMITATIONS.strip()
